<a href="https://colab.research.google.com/github/venkata18167/CSA6301---THREAT-INTELLIGENCE-AND-NETWORK-SECURITY/blob/main/20_Multi_Source_Log_Correlation_for_Brute_Force_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
 import pandas as pd

print("="*70)
print(" MULTI-SOURCE LOG CORRELATION FOR BRUTE-FORCE DETECTION ")
print("="*70)


firewall_logs = pd.DataFrame({
    "Timestamp":[
        "10:00:01","10:00:05","10:00:08",
        "10:00:10","10:00:15","10:00:20"
    ],
    "IP":[
        "192.168.1.10",
        "192.168.1.20",
        "192.168.1.10",
        "192.168.1.30",
        "192.168.1.10",
        "192.168.1.20"
    ],
    "Firewall_Status":[
        "Allowed",
        "Allowed",
        "Allowed",
        "Allowed",
        "Allowed",
        "Blocked"
    ]
})


ssh_logs = pd.DataFrame({
    "Timestamp":[
        "10:00:02","10:00:04","10:00:06",
        "10:00:09","10:00:11","10:00:16",
        "10:00:21","10:00:25"
    ],
    "IP":[
        "192.168.1.10",
        "192.168.1.10",
        "192.168.1.10",
        "192.168.1.20",
        "192.168.1.10",
        "192.168.1.30",
        "192.168.1.10",
        "192.168.1.20"
    ],
    "SSH_Status":[
        "Failed",
        "Failed",
        "Failed",
        "Success",
        "Failed",
        "Failed",
        "Failed",
        "Success"
    ]
})

web_logs = pd.DataFrame({
    "Timestamp":[
        "10:00:03","10:00:07","10:00:12",
        "10:00:18","10:00:22"
    ],
    "IP":[
        "192.168.1.10",
        "192.168.1.20",
        "192.168.1.10",
        "192.168.1.30",
        "192.168.1.10"
    ],
    "Web_Request":[
        "Login",
        "Home",
        "Login",
        "Login",
        "Login"
    ]
})


print("\nFirewall Logs\n")
print(firewall_logs)

print("\nSSH Logs\n")
print(ssh_logs)

print("\nWeb Server Logs\n")
print(web_logs)


failed_attempts = ssh_logs[ssh_logs["SSH_Status"]=="Failed"]

count = failed_attempts.groupby("IP").size().reset_index(name="Failed_Logins")

merged = pd.merge(count,
                  firewall_logs.groupby("IP").size().reset_index(name="Firewall_Events"),
                  on="IP",
                  how="left")

merged = pd.merge(merged,
                  web_logs.groupby("IP").size().reset_index(name="Web_Requests"),
                  on="IP",
                  how="left")

merged.fillna(0,inplace=True)


THRESHOLD = 5

status=[]
risk=[]

for attempts in merged["Failed_Logins"]:

    if attempts >= THRESHOLD:
        status.append("Brute Force Suspected")
        risk.append("HIGH")
    else:
        status.append("Normal")
        risk.append("LOW")

merged["Status"]=status
merged["Risk_Level"]=risk


print("\n")
print("="*70)
print("CORRELATED SECURITY REPORT")
print("="*70)

print(merged)

print("\n")
print("="*70)
print("ALERTS")
print("="*70)

alerts = merged[merged["Status"]=="Brute Force Suspected"]

if len(alerts)==0:
    print("No Brute Force Attack Detected.")
else:
    for index,row in alerts.iterrows():
        print("IP Address       :",row["IP"])
        print("Failed Logins    :",row["Failed_Logins"])
        print("Firewall Events  :",row["Firewall_Events"])
        print("Web Requests     :",row["Web_Requests"])
        print("Risk Level       :",row["Risk_Level"])
        print("-"*50)


filename="Brute_Force_Report.csv"

merged.to_csv(filename,index=False)

print("\nReport Saved As :",filename)

print("\nProgram Completed Successfully.")

 MULTI-SOURCE LOG CORRELATION FOR BRUTE-FORCE DETECTION 

Firewall Logs

  Timestamp            IP Firewall_Status
0  10:00:01  192.168.1.10         Allowed
1  10:00:05  192.168.1.20         Allowed
2  10:00:08  192.168.1.10         Allowed
3  10:00:10  192.168.1.30         Allowed
4  10:00:15  192.168.1.10         Allowed
5  10:00:20  192.168.1.20         Blocked

SSH Logs

  Timestamp            IP SSH_Status
0  10:00:02  192.168.1.10     Failed
1  10:00:04  192.168.1.10     Failed
2  10:00:06  192.168.1.10     Failed
3  10:00:09  192.168.1.20    Success
4  10:00:11  192.168.1.10     Failed
5  10:00:16  192.168.1.30     Failed
6  10:00:21  192.168.1.10     Failed
7  10:00:25  192.168.1.20    Success

Web Server Logs

  Timestamp            IP Web_Request
0  10:00:03  192.168.1.10       Login
1  10:00:07  192.168.1.20        Home
2  10:00:12  192.168.1.10       Login
3  10:00:18  192.168.1.30       Login
4  10:00:22  192.168.1.10       Login


CORRELATED SECURITY REPORT
             I